# Hasil pilot People Also Ask

Notebook ini hanya membaca hasil lokal, **tidak memanggil API**. Pilih kernel `venv` proyek dan jalankan sel berurutan. Pengambilan dilakukan melalui `src/collect_paa.py`; petunjuk ada di `docs/PAA_COLLECTION.md`.

Batch 15 topik dipilih secara purposif untuk eksplorasi bimbingan. Pertanyaan PAA masih perlu ditinjau sebelum sintesis; ini belum dataset utama pasangan query–artikel.

In [2]:
from pathlib import Path
import csv
import json
from collections import Counter
from html import escape
from IPython.display import HTML, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/collect_paa.py').exists())
BATCH_ID = 'paa_pilot_01'
BASE = ROOT / 'data/interim/paa' / BATCH_ID

def read_table(name):
    with (BASE / name).open(encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))

def show(rows, columns):
    header = ''.join('<th>' + escape(c) + '</th>' for c in columns)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(r.get(c, ''))) + '</td>' for c in columns) + '</tr>' for r in rows)
    display(HTML('<div style="overflow-x:auto"><table><tr>' + header + '</tr>' + body + '</table></div>'))

searches = read_table('searches.csv')
questions = read_table('questions.csv')
print('Catatan pencarian:', len(searches))
print('Status:', dict(Counter(r['status'] for r in searches)))
print('Kemunculan pertanyaan:', len(questions))
print('Teks pertanyaan unik setelah normalisasi:', len({r['question_key'] for r in questions}))

Catatan pencarian: 15
Status: {'success': 13, 'no_paa': 2}
Kemunculan pertanyaan: 52
Teks pertanyaan unik setelah normalisasi: 52


## Ringkasan per domain

`success` berarti respons memuat pertanyaan PAA; `no_paa` berarti respons berhasil tanpa pertanyaan. Keduanya tetap dilaporkan. `not_requested`, `started`, dan `error` belum menghasilkan observasi lengkap.

In [3]:
summary = []
for domain in ['kesehatan', 'keuangan', 'teknologi']:
    group = [r for r in searches if r['research_domain'] == domain]
    q = [r for r in questions if r['research_domain'] == domain]
    summary.append({'domain': domain, 'topik': len(group), 'dengan_paa': sum(r['status'] == 'success' for r in group),
                    'tanpa_paa': sum(r['status'] == 'no_paa' for r in group), 'pertanyaan': len(q),
                    'pertanyaan_unik': len({r['question_key'] for r in q})})
show(summary, ['domain', 'topik', 'dengan_paa', 'tanpa_paa', 'pertanyaan', 'pertanyaan_unik'])
show(searches, ['research_domain', 'retrieval_query', 'status', 'question_count', 'retrieved_at'])

domain,topik,dengan_paa,tanpa_paa,pertanyaan,pertanyaan_unik
kesehatan,5,5,0,20,20
keuangan,5,5,0,20,20
teknologi,5,3,2,12,12


research_domain,retrieval_query,status,question_count,retrieved_at
kesehatan,icd 10,success,4,2026-09-10T04:42:07.681301+00:00
kesehatan,obat batuk,success,4,2026-09-10T04:42:09.468561+00:00
kesehatan,campak,success,4,2026-09-10T04:42:15.655108+00:00
kesehatan,vitamin c,success,4,2026-09-10T04:42:20.415934+00:00
kesehatan,sunscreen,success,4,2026-09-10T04:42:22.101471+00:00
keuangan,pajak,success,4,2026-09-10T04:42:23.877289+00:00
keuangan,npwp,success,4,2026-09-10T04:42:29.986171+00:00
keuangan,pegadaian,success,4,2026-09-10T04:42:32.549389+00:00
keuangan,pinjaman daring,success,4,2026-09-10T04:42:42.387145+00:00
keuangan,dividen,success,4,2026-09-10T04:42:45.611171+00:00


## Periksa pertanyaan dan asalnya

Ubah `DOMAIN`, `KEYWORD`, dan `START` untuk meninjau halaman lain. Pertanyaan yang muncul untuk suatu topik belum tentu sesuai domain. Bandingkan pertanyaan dengan topik sumber dan tandai rencana keputusan dalam catatan terpisah; notebook ini belum menyimpan keputusan PAA.

Perhatikan pula hasil rekomendasi produk, pertanyaan berbahasa lain, duplikasi, dan pertanyaan yang menyimpang. Jangan mengubah pertanyaan agar terlihat relevan.

In [7]:
DOMAIN = 'teknologi'  # 'kesehatan', 'keuangan', 'teknologi', atau None
KEYWORD = ''
START = 0
PAGE_SIZE = 20
selected = [r for r in questions if (DOMAIN is None or r['research_domain'] == DOMAIN)
            and KEYWORD.casefold() in (r['question'] + ' ' + r['retrieval_query']).casefold()]
show(selected[START:START+PAGE_SIZE], ['paa_id', 'research_domain', 'retrieval_query', 'question', 'position', 'result_type', 'topic_id'])
print('Jumlah sesuai filter:', len(selected))

paa_id,research_domain,retrieval_query,question,position,result_type,topic_id
paa_0792b4e3301f2eb2227f5be3,teknologi,vpn,VPN itu buat apa sih?,1,ai_overview,topic_ebf20cefc9169e0b
paa_42d2ba0b949326fa267edd94,teknologi,vpn,Cara mengaktifkan VPN gimana?,2,ai_overview,topic_ebf20cefc9169e0b
paa_3c425217dc5465eb3106f4dd,teknologi,vpn,Apa nama VPN gratis?,3,ai_overview,topic_ebf20cefc9169e0b
paa_9046c16081e3bff38ff46a02,teknologi,vpn,Apa resiko VPN gratis?,4,ai_overview,topic_ebf20cefc9169e0b
paa_040e0ad89f7c27cc26cd91b7,teknologi,excel,Rumus apa saja di Excel?,1,ai_overview,topic_faba1e00af8e6c89
paa_c2422918a74ab63d944a6ad3,teknologi,excel,Apa sih Excel itu?,2,ai_overview,topic_faba1e00af8e6c89
paa_6a9b199e96ff63f41113cb0c,teknologi,excel,Apakah Excel bisa di HP?,3,ai_overview,topic_faba1e00af8e6c89
paa_44ccc1a60118134b625f890a,teknologi,excel,Fungsi Excel apa saja?,4,ai_overview,topic_faba1e00af8e6c89
paa_4efc9d286845a7e311d8ba7c,teknologi,laptop asus,Laptop ASUS harga berapa?,1,ai_overview,topic_c99910eca7614b65
paa_d04edf154cb087525edb7504,teknologi,laptop asus,Laptop ASUS ram 4 harganya berapa?,2,ai_overview,topic_c99910eca7614b65


Jumlah sesuai filter: 12


## Kuota dan langkah berikutnya

Baca snapshot kuota yang sudah disimpan; tidak ada panggilan Account API dari notebook. Setelah peninjauan, pertanyaan yang diterima dapat digabung dengan sumber Trends siap sintesis. Nilai prompt melalui Judgment pertama sebelum menjalankan sintesis.

In [4]:
quota_dir = ROOT / 'data/raw/paa/batches' / BATCH_ID
for path in sorted(quota_dir.glob('quota_*.json')):
    snapshot = json.loads(path.read_text(encoding='utf-8'))
    print(path.name, snapshot)
print('Dataset pertanyaan:', BASE / 'questions.csv')

quota_after.json {'checked_at': '2026-09-10T04:43:18.879948+00:00', 'total_searches_left': 22, 'plan_searches_left': 22, 'searches_per_month': 250, 'this_month_usage': 228}
quota_before_20260910T044207680300Z.json {'checked_at': '2026-09-10T04:42:07.679301+00:00', 'total_searches_left': 37, 'plan_searches_left': 37, 'searches_per_month': 250, 'this_month_usage': 213}
Dataset pertanyaan: d:\Kuliah\TA\final-assignment\data\interim\paa\paa_pilot_01\questions.csv
